In [ ]:
from dataclasses import dataclass, field
from pydantic import BaseModel, field_validator, ValidationError
import re

ASSISTANT_NAME = "Risk Management"
SERVICES = ["risk_management", "risk_identification", "risk_mitigation", "other"]
CITIES = ["Riyadh", "Jeddah", "Makkah", "Dammam", "unknown"]
print(ASSISTANT_NAME, SERVICES)

Risk Management ['risk_management', 'risk_identification', 'risk_mitigation', 'other']


In [ ]:
class ServiceTicket(BaseModel):
    full_name: str
    service_type: str
    city: str
    urgent: bool
    national_id: str | None = None

    @field_validator("service_type")
    @classmethod
    def check_service(cls, value):
        if value not in SERVICES:
            raise ValueError("service_type is not supported")
        return value

    @field_validator("city")
    @classmethod
    def check_city(cls, value):
        if value not in CITIES:
            raise ValueError("city is not supported")
        return value

    @field_validator("national_id")
    @classmethod
    def check_national_id(cls, value):
        if value is not None and (len(value) != 10 or not value.isdigit() or value[0] not in "12"):
            raise ValueError("invalid national_id")
        return value

ticket = ServiceTicket(full_name="Ahmed", service_type="risk_management", city="Riyadh", urgent=True)
print(ticket)

full_name='Ahmed' service_type='risk_management' city='Riyadh' urgent=True national_id=None


In [ ]:
@dataclass
class LLMRequest:
    messages: list[dict]
    model: str
    max_tokens: int = 512

@dataclass
class LLMResponse:
    text: str

class DemoLLM:
    def complete(self, request: LLMRequest):
        return LLMResponse(f"Risk Management response to: {request.messages[-1]['content']}")

@dataclass
class ConversationState:
    max_turns: int = 8
    messages: list[dict] = field(default_factory=list)

    def add(self, role, content):
        self.messages.append({"role": role, "content": content})
        self.messages = self.messages[-(self.max_turns * 2):]

    def get_messages(self):
        return self.messages

def chat(conversation, client, message):
    conversation.add("user", message)
    response = client.complete(LLMRequest(conversation.get_messages(), "demo-model"))
    conversation.add("assistant", response.text)
    return response.text

conversation = ConversationState()
print(chat(conversation, DemoLLM(), "What is risk management?"))

Risk Management response to: What is risk management?


In [ ]:
BLOCKED_PHRASES = ["تجاهل التعليمات", "ignore previous instructions", "system prompt"]
SCOPE_WORDS = ["حوادث", "تهديد", "تقييم", "مخاطر"]
PII_PATTERNS = {"NATIONAL_ID": r"\b[12]\d{9}\b", "PHONE": r"\b05\d{8}\b"}
SECRET_CODE = "Risk-2026"

def check_input(message):
  if any(p in message for p in BLOCKED_PHRASES):
     return False, "injection"
  if any(w in message for w in SCOPE_WORDS):
     return True, "ok"
  return False, "off_scope"

def mask_pii(message):
  for name, pattern in PII_PATTERNS.items():
   message = re.sub(pattern, f"[{name}]", message)
  return message

def check_output(answer):
  return (False, "leak") if SECRET_CODE in answer else (True, "ok")

print(mask_pii("هويتي 1012345678 وجوالي 0551234567"))
print(check_input("كيف أقيم المخاطر؟"))

هويتي [NATIONAL_ID] وجوالي [PHONE]
(True, 'ok')


In [ ]:
CURRENT_USER = "Ahmed"
APPLICATIONS = {"CR12345678": "under review", "CR87654321": "approved"}

def check_risk_status(reference):
    return APPLICATIONS.get(reference, "not found")

def schedule_risk_assessment(name, city, date):
    return f"Assessment scheduled for {name} in {city} on {date}"

TOOLS = {
    "check_risk_status": {"function": check_risk_status, "risk": "read_only"},
    "schedule_risk_assessment": {"function": schedule_risk_assessment, "risk": "side_effecting"}
}

def run_tool(name, args):
    if name not in TOOLS:
        return "error: unknown tool"
    tool = TOOLS[name]
    if tool["risk"] == "side_effecting" and args.get("name") != CURRENT_USER:
        return "error: not authorized"
    try:
        return tool["function"](**args)
    except TypeError:
        return "error: wrong arguments"

print(run_tool("check_risk_status", {"reference": "CR12345678"}))
print(run_tool("schedule_risk_assessment", {"name": "Ahmed", "city": "Riyadh", "date": "2026-10-10"}))

under review
Assessment scheduled for Ahmed in Riyadh on 2026-10-10


In [ ]:
GOLDEN_SET = [
    ("كيف أقيم المخاطر في المشروع؟", "تقييم المخاطر"),
    ("ماهي استراتيجيات التعامل مع المخاطر؟", "التجنب"),
    ("وش أفضل مطعم في الرياض؟", "خارج خدماتنا")
]

def answer_question(question):
    if "أقيم" in question:
        return "تقييم المخاطر يتم عبر مصفوفة تقييم المخاطر"
    if "استراتيجيات" in question:
        return "التجنب، التخفيف، نقل الخطر، القبول"
    return "هذا خارج خدماتنا."

def evaluate():
    passed = sum(expected in answer_question(q) for q, expected in GOLDEN_SET)
    return passed * 100 // len(GOLDEN_SET)

print("Evaluation score:", evaluate(), "%")

Evaluation score: 100 %


In [ ]:
PRICE = {"flagship": 10, "cheap": 2}
COMPLEX_WORDS = ["اعتراض", "شكوى"]

def choose_model(message):
    return "flagship" if any(w in message for w in COMPLEX_WORDS) else "cheap"

def run_request(message, cache):
    if message in cache:
        return cache[message], "cache"
    model = choose_model(message)
    answer = "تم استلام طلبك وسيتم الرد عليك."
    cache[message] = answer
    return answer, model

cache = {}
for message in ["كيف أقدم بلاغ عن خطر جديد؟", "أبي أقدم اعتراض على تقرير مخاطر الامتثال", "كيف أقدم بلاغ عن خطر جديد؟"]:
    print(run_request(message, cache))

('تم استلام طلبك وسيتم الرد عليك.', 'cheap')
('تم استلام طلبك وسيتم الرد عليك.', 'flagship')
('تم استلام طلبك وسيتم الرد عليك.', 'cache')
